# 3 — Database setup (SQLite — no server needed)

This notebook creates the SQLite database and loads the cleaned CSVs.

SQLite runs entirely in-process — no server, no credentials, no `psycopg2`.
The database file is stored at `../data/healthcare.db`.

Once this is done, all SQL analysis runs against a consistent schema and reusable joined view (`vw_inpatient_analytics`).

## SQLite — no server required

SQLite 3.25+ supports all window functions used in this project:
`RANK()`, `DENSE_RANK()`, `ROW_NUMBER()`, `PERCENT_RANK()`, `SUM() OVER (...)`.

The database file is created automatically on first run.

In [8]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

DB_PATH = (Path('..') / 'data' / 'healthcare.db').resolve()
engine = create_engine(f'sqlite:///{DB_PATH}')
print(f'SQLite database: {DB_PATH}')
engine

SQLite database: /Users/apekshaa/Desktop/Healthcare_Analysis-main/data/healthcare.db


Engine(sqlite:////Users/apekshaa/Desktop/Healthcare_Analysis-main/data/healthcare.db)

## Create tables

The schema lives in `sql/create_tables.sql`. Keeping it there makes it easy to review and keeps types consistent across runs.


In [9]:
create_sql = (Path('..') / 'sql' / 'create_tables.sql').read_text(encoding='utf-8')
with engine.begin() as conn:
    for stmt in [s.strip() for s in create_sql.split(';') if s.strip() and not s.strip().startswith('--')]:
        conn.execute(text(stmt))
print('Tables created (or already exist).')

Tables created (or already exist).


## Load cleaned CSVs into PostgreSQL

I use `pandas.to_sql` here for portability. It’s not the fastest option in the world, but it’s reliable for a local case study.


In [10]:
CLEANED = Path('..') / 'data' / 'cleaned'
inpatient = pd.read_csv(CLEANED / 'inpatient_payments_cleaned.csv')
provider = pd.read_csv(CLEANED / 'provider_info_cleaned.csv')
drg = pd.read_csv(CLEANED / 'drg_details_cleaned.csv')

inpatient.to_sql('inpatient_payments', engine, if_exists='replace', index=False, chunksize=5000)
provider.to_sql('provider_info', engine, if_exists='replace', index=False, chunksize=5000)
drg.to_sql('drg_details', engine, if_exists='replace', index=False, chunksize=5000)

print('Loaded rows:')
print('inpatient_payments', len(inpatient))
print('provider_info', len(provider))
print('drg_details', len(drg))


Loaded rows:
inpatient_payments 196086
provider_info 5426
drg_details 518


## Create the joined analytical view

Instead of repeating joins in every query, I create a single view `vw_inpatient_analytics`.
That makes the SQL notebook easier to read and reduces the chance of inconsistent join logic.


In [12]:
join_sql = (Path('..') / 'sql' / 'joins.sql').read_text(encoding='utf-8')
with engine.begin() as conn:
    for stmt in [s.strip() for s in join_sql.split(';') if s.strip() and not s.strip().startswith('--')]:
        conn.execute(text(stmt))
print('View created: vw_inpatient_analytics')

OperationalError: (sqlite3.OperationalError) view vw_inpatient_analytics already exists
[SQL: CREATE VIEW vw_inpatient_analytics AS
SELECT
    ip.facility_id,
    ip.rndrng_prvdr_org_name      AS provider_name,
    ip.rndrng_prvdr_state_abrvtn  AS state,
    ip.drg_cd,
    ip.drg_desc,
    ip.tot_dschrgs,
    ip.avg_submtd_cvrd_chrg,
    ip.avg_mdcr_pymt_amt,
    ip.billing_gap,
    ip.charge_ratio,
    p.hospital_type,
    p.hospital_ownership,
    d.mdc,
    d.mdc_name,
    d.drg_type,
    d.relative_weights,
    d.geometric_mean_los
FROM inpatient_payments ip
LEFT JOIN provider_info p
    ON ip.facility_id = p.facility_id
LEFT JOIN drg_details d
    ON ip.drg_cd = d.drgv22]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## Verify loads

A quick row count check is a simple way to avoid wasting time later when a load silently failed.


In [6]:
with engine.begin() as conn:
    rows = conn.execute(text('''
        SELECT 'inpatient_payments' AS tbl, COUNT(*) AS n FROM inpatient_payments
        UNION ALL
        SELECT 'provider_info', COUNT(*) FROM provider_info
        UNION ALL
        SELECT 'drg_details', COUNT(*) FROM drg_details
        UNION ALL
        SELECT 'vw_inpatient_analytics', COUNT(*) FROM vw_inpatient_analytics;
    ''')).fetchall()
rows


[('inpatient_payments', 196086),
 ('provider_info', 5426),
 ('drg_details', 518),
 ('vw_inpatient_analytics', 196086)]